In [ ]:
# --- edit these two paths ---
FLYBODY_SRC = "/path/to/local/flybody-main"
NEURO_FLY   = "/path/to/local/digifly-legacy/Python Code/Phase 3 - NEURON-to-MuJoCo Bridge/neuro_fly"   # root of the unzipped repo

import sys, os
from pathlib import Path

def add_if_missing(p: Path):
    p = p.resolve()
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# 1) Make flybody importable
add_if_missing(Path(FLYBODY_SRC))
try:
    import flybody.fly_envs as fb
except Exception as e:
    print("[flybody] import failed:", e)

# 2) Find the neuro_adapter package inside NEURO_FLY
root = Path(NEURO_FLY).resolve()
candidates = []
for pat in ["neuro_adapter/__init__.py", "src/neuro_adapter/__init__.py"]:
    hit = root / pat
    if hit.exists():
        # we must add the *parent of 'neuro_adapter'* to sys.path
        candidates.append(hit.parent.parent)

if not candidates:
    # last resort: scan a bit deeper
    for hit in root.rglob("__init__.py"):
        if hit.parent.name == "neuro_adapter":
            candidates.append(hit.parent.parent)
            break

if not candidates:
    raise RuntimeError("Could not locate a 'neuro_adapter' package inside NEURO_FLY; "
                       "make sure the zip contains a 'neuro_adapter' folder (possibly under 'src/').")

# prefer a 'src/' candidate if present
cand = sorted(candidates, key=lambda p: (0 if p.name == "src" else 1, len(str(p))))[0]
add_if_missing(cand)

# 3) Try imports now
import neuro_adapter
from neuro_adapter import env_loader
from neuro_adapter.env_loader import load_env, list_actuator_names
print("Imports OK ✅ from", cand)


In [ ]:
!python -m pip install -q numpy pyyaml imageio imageio-ffmpeg mujoco dm-control

In [ ]:
import inspect, flybody.fly_envs as fb
print("WalkImitation signature:", inspect.signature(fb.WalkImitation.__init__))
candidates = [n for n in dir(fb) if "mocap" in n.lower() or "traj" in n.lower()]
print("Possible helpers in fly_envs:", candidates)


In [ ]:
# Replace env_loader.load_env to use the factory functions in flybody.fly_envs
import types
import neuro_adapter.env_loader as el
import flybody.fly_envs as fb

def load_env_patched(mode: str = "walk", **kwargs):
    m = mode.lower()
    if m in ("walk", "walking", "walk_imitation"):
        # Inference mode if ref_path is not provided
        return fb.walk_imitation(**kwargs)                 # accepts ref_path=None, etc.
    if m in ("walk_on_ball", "ball"):
        return fb.walk_on_ball(**kwargs)
    if m in ("flight", "fly", "flight_wpg"):
        return fb.flight_imitation(**kwargs)               # accepts ref_path=None, etc.
    if hasattr(fb, "make_env"):
        return fb.make_env(mode, **kwargs)
    raise ValueError(f"Unknown mode '{mode}'")

el.load_env = load_env_patched
print("Patched env loader ✅")


In [ ]:
from neuro_adapter.env_loader import load_env, list_actuator_names

env = load_env(mode="walk")       # uses fb.walk_imitation(ref_path=None) → inference loader
names = list_actuator_names(env)
print(f"{len(names)} actuators")
for n in names[:40]:
    print(" -", n)


In [ ]:
from pathlib import Path
import yaml, os

mapping_yaml = Path(NEURO_FLY) / "neuro_adapter" / "mapping.yaml"
mapping = {
    "global": {"neural_dt_ms": 0.025, "clip_actions": True},
    "actuators": [
        # Use names from your printout:
        {"pool": "L1_coxa_pitch_ext",  "actuator": "walker/coxa_T1_left",   "type": "torque", "gain": 0.8, "sign":  1.0},
        {"pool": "L1_coxa_pitch_flex", "actuator": "walker/coxa_T1_left",   "type": "torque", "gain": 0.8, "sign": -1.0},

        {"pool": "L1_femur_drive",     "actuator": "walker/femur_T1_left",  "type": "torque", "gain": 0.6, "sign":  1.0},
        {"pool": "L1_tibia_drive",     "actuator": "walker/tibia_T1_left",  "type": "torque", "gain": 0.6, "sign":  1.0},

        {"pool": "abdomen_drive",      "actuator": "walker/abdomen",        "type": "torque", "gain": 0.4, "sign":  1.0},
        {"pool": "head_drive",         "actuator": "walker/head",           "type": "torque", "gain": 0.2, "sign":  1.0},
    ]
}
with open(mapping_yaml, "w") as f:
    yaml.safe_dump(mapping, f, sort_keys=False)
print("Wrote", mapping_yaml)


In [ ]:
# 0) Pick a good camera and narrow its FOV (zoom-in helper)
import numpy as np

def set_fov(env, cam, fovy_deg=20.0):
    ph = env.physics
    # resolve name -> index
    idx = None
    if isinstance(cam, str):
        for i in range(ph.model.ncam):
            if ph.model.id2name(i, 'camera') == cam:
                idx = i; break
        if idx is None:
            raise ValueError(f"Camera '{cam}' not found.")
    else:
        idx = int(cam)
    ph.model.cam_fovy[idx] = float(fovy_deg)

cam_name = 'walker/hero'  # from your list

# 1) Create a NEW adapter each run (do NOT reuse an old one)
from neuro_adapter.adapter import NeuroMuJoCoAdapter
from neuron_bridge.bridge import NeuronBridge
import yaml, pathlib

mapping_yaml = pathlib.Path(NEURO_FLY) / "neuro_adapter" / "mapping.yaml"
stim_yaml    = pathlib.Path(NEURO_FLY) / "examples" / "stim_walk.yaml"
video_path   = str(pathlib.Path(NEURO_FLY) / "outputs" / "walk_demo_close.mp4")

bridge = NeuronBridge({})
adapter = NeuroMuJoCoAdapter(
    mode="walk",                                # or "walk_on_ball"
    mapping=yaml.safe_load(open(mapping_yaml)),
    bridge=bridge,
    render_fps=60,
    video_path=video_path,
    render_size=(720, 720),
)

# 2) Zoom the chosen camera before the run
set_fov(adapter.env, cam_name, fovy_deg=18.0)  # try 18–25° for a nice close shot

# 3) Run ONCE (don’t call adapter.run twice on the same adapter)
adapter.run(
    stim_plan=yaml.safe_load(open(stim_yaml)),
    duration_s=5.0,
    render_cam=cam_name,         # use a named camera you listed
)

print("Episode complete:", video_path)


In [ ]:
from IPython.display import HTML
from base64 import b64encode

with open(video_path, "rb") as f:
    data_url = "data:video/mp4;base64," + b64encode(f.read()).decode()
HTML(f"""
<video width="720" controls>
  <source src="{data_url}" type="video/mp4">
</video>
""")


In [ ]:
from neuro_adapter.env_loader import load_env, list_actuator_names
env = load_env("walk")  # or "walk_on_ball"
names = [n for n in list_actuator_names(env)]
legs = [n for n in names if "walker/" in n and any(k in n for k in ["coxa_", "femur_", "tibia_"])]
print(len(legs), "leg actuators"); print("\n".join(legs))
adh = [n for n in names if "adh" in n.lower() or "grip" in n.lower() or "adhesion" in n.lower()]
print("adhesion-like:", adh)


In [ ]:
# TripodCPGBridge: generates coordinated rates for coxa (swing), femur/tibia (lift/place)
import math
from neuron_bridge import bridge as bridge_mod

class TripodCPGBridge(bridge_mod.DummyBridge):
    def __init__(self, freq_hz=2.0, amp=1.0):
        super().__init__({})
        self.freq = float(freq_hz)
        self.amp  = float(amp)
        self._pools = {}          # filled by set_leg_pools()
        self._pools_known = set() # ensures adapter validation is happy

    def set_leg_pools(self, pools):
        """
        pools format per leg label (L1,R1,L2,R2,L3,R3):
          {"L1": {"coxa":"...", "femur":"...", "tibia":"..."},
           "R1": {...}, ...}
        """
        self._pools = pools
        self._pools_known.update([p for leg in pools.values() for p in leg.values() if p])

    def _rate(self, phase, t_ms):
        # half-wave rectified sin → 0..1
        x = math.sin(2*math.pi*self.freq*(t_ms/1000.0) + phase)
        return self.amp * max(0.0, x)

    def readout(self, pool_names):
        # tripod grouping
        tripodA = {"L1","R2","L3"}
        t = self._t_ms
        out = {}
        for leg, joints in self._pools.items():
            isA = leg in tripodA
            base = 0.0 if isA else math.pi     # A vs B 180° out of phase

            # Phase scheduling:
            # - coxa: swing timing (base)
            # - femur: starts lift a bit before swing (lead 60°)
            # - tibia: peaks earlier to lift, then extend near stance end (lead 90°)
            phases = {
                "coxa":  base + 0.0,
                "femur": base + math.pi/3,
                "tibia": base + math.pi/2,
            }

            for joint, pool in joints.items():
                if not pool: continue
                hz = 140.0 * self._rate(phases[joint], t)   # 0..~140 Hz; adapter low-pass makes it smooth
                out[pool] = {"mode":"rate", "value": hz}

        # default zeros for any unmapped pool the adapter asks about
        for p in pool_names:
            out.setdefault(p, {"mode":"rate","value":0.0})
        return out

# Make this the exported bridge the adapter will import
bridge_mod.NeuronBridge = TripodCPGBridge
print("TripodCPGBridge installed ✅")


In [ ]:
from pathlib import Path
import yaml

# Your actuator lists from the env
leg_actuators = [
"walker/coxa_abduct_T1_left","walker/coxa_twist_T1_left","walker/coxa_T1_left",
"walker/femur_twist_T1_left","walker/femur_T1_left","walker/tibia_T1_left",
"walker/coxa_abduct_T1_right","walker/coxa_twist_T1_right","walker/coxa_T1_right",
"walker/femur_twist_T1_right","walker/femur_T1_right","walker/tibia_T1_right",
"walker/coxa_abduct_T2_left","walker/coxa_twist_T2_left","walker/coxa_T2_left",
"walker/femur_twist_T2_left","walker/femur_T2_left","walker/tibia_T2_left",
"walker/coxa_abduct_T2_right","walker/coxa_twist_T2_right","walker/coxa_T2_right",
"walker/femur_twist_T2_right","walker/femur_T2_right","walker/tibia_T2_right",
"walker/coxa_abduct_T3_left","walker/coxa_twist_T3_left","walker/coxa_T3_left",
"walker/femur_twist_T3_left","walker/femur_T3_left","walker/tibia_T3_left",
"walker/coxa_abduct_T3_right","walker/coxa_twist_T3_right","walker/coxa_T3_right",
"walker/femur_twist_T3_right","walker/femur_T3_right","walker/tibia_T3_right",
]
adhesion = [
"walker/adhere_claw_T1_left","walker/adhere_claw_T1_right",
"walker/adhere_claw_T2_left","walker/adhere_claw_T2_right",
"walker/adhere_claw_T3_left","walker/adhere_claw_T3_right",
]

# Helper to get actuator by pattern
def find_name(substr):
    for n in leg_actuators:
        if substr in n:
            return n
    return None

legs = [
    ("L1","T1","left"), ("R1","T1","right"),
    ("L2","T2","left"), ("R2","T2","right"),
    ("L3","T3","left"), ("R3","T3","right"),
]

actuators = []
for label, thorax, side in legs:
    # use the pitch-ish joints: coxa_T*, femur_T*, tibia_T*
    coxa  = find_name(f"walker/coxa_{thorax}_{side}")
    femur = find_name(f"walker/femur_{thorax}_{side}")
    tibia = find_name(f"walker/tibia_{thorax}_{side}")
    if coxa:
        actuators += [
            {"pool": f"{label}_coxa_ext",  "actuator": coxa,  "type":"torque", "gain":0.20, "sign": +1.0},
            {"pool": f"{label}_coxa_flex", "actuator": coxa,  "type":"torque", "gain":0.20, "sign": -1.0},
        ]
    if femur:
        actuators.append({"pool": f"{label}_femur_drive","actuator": femur,"type":"torque","gain":0.15,"sign": +1.0})
    if tibia:
        actuators.append({"pool": f"{label}_tibia_drive","actuator": tibia,"type":"torque","gain":0.15,"sign": +1.0})

# adhesion as gentle normal-force
for a in adhesion:
    tag = a.split("/")[-1]
    actuators.append({"pool": f"adh_{tag}", "actuator": a, "type":"adhesion",
                      "k": 10.0, "x0": 0.35, "fmax": 3.0})

mapping = {"global":{"neural_dt_ms":0.025,"clip_actions":True},"actuators":actuators}
mpath = Path(NEURO_FLY) / "neuro_adapter" / "mapping.yaml"
with open(mpath,"w") as f:
    yaml.safe_dump(mapping, f, sort_keys=False)

print("Wrote mapping with", len(actuators), "entries →", mpath)


In [ ]:
import yaml, pathlib, numpy as np
from neuron_bridge.bridge import NeuronBridge          # now TripodCPGBridge
from neuro_adapter.adapter import NeuroMuJoCoAdapter

mapping_yaml = pathlib.Path(NEURO_FLY) / "neuro_adapter" / "mapping.yaml"
mp = yaml.safe_load(open(mapping_yaml))

# Tell the CPG which pool names correspond to which joints (so it drives the right ones)
bridge = NeuronBridge(freq_hz=2.2, amp=1.0)
pools = {}
for spec in mp["actuators"]:
    pool = spec["pool"]
    if "_coxa_" in pool:
        leg = pool.split("_coxa_")[0]  # e.g., "L1"
        pools.setdefault(leg, {})["coxa"] = pool
    if "_femur_" in pool:
        leg = pool.split("_femur_")[0]
        pools.setdefault(leg, {})["femur"] = pool
    if "_tibia_" in pool:
        leg = pool.split("_tibia_")[0]
        pools.setdefault(leg, {})["tibia"] = pool
bridge.set_leg_pools(pools)

# Fresh adapter each run
video_path = str(pathlib.Path(NEURO_FLY)/"outputs"/"walk_tripod_full.mp4")
adapter = NeuroMuJoCoAdapter(mode="walk", mapping=mp, bridge=bridge,
                             render_fps=60, video_path=video_path, render_size=(720,720))

# Zoom the camera so it isn't tiny
def set_fov(env, cam_name, fovy_deg=18.0):
    ph = env.physics
    idx = [i for i in range(ph.model.ncam) if ph.model.id2name(i,'camera')==cam_name][0]
    ph.model.cam_fovy[idx] = float(fovy_deg)

cam = "walker/hero"               # from your list; 'walker/side' is also good
set_fov(adapter.env, cam, fovy_deg=18.0)

# Run 6–10 s to see a few gait cycles
stim_yaml = pathlib.Path(NEURO_FLY)/"examples"/"stim_walk.yaml"
adapter.run(stim_plan=yaml.safe_load(open(stim_yaml)), duration_s=8.0, render_cam=cam)
print("Video:", video_path)


In [ ]:
# --- Antagonist-aware tripod CPG that also drives adhesion in stance ---
import math, re, yaml, pathlib
from neuron_bridge import bridge as bridge_mod

class TripodCPGBridgeAF(bridge_mod.DummyBridge):
    """
    Drives per-leg antagonists (coxa_ext / coxa_flex) + femur / tibia and adhesion.
    - Tripod A: L1, R2, L3 (phase 0)
      Tripod B: R1, L2, R3 (phase π)
    - coxa_ext ~ swing (phase 0), coxa_flex ~ stance (phase π)
    - femur/tibia lead coxa to lift/place foot
    - adhesion high in stance, low in swing
    """
    def __init__(self, freq_hz=2.0, amp=1.0):
        super().__init__({})
        self.freq = float(freq_hz)
        self.amp  = float(amp)
        self.legs = {}         # {"L1":{"coxa_ext":..., "coxa_flex":..., "femur":..., "tibia":..., "adh":...}, ...}
        self._pools_known = set()

    def set_mapping_from_yaml(self, mapping_yaml_path: str):
        mp = yaml.safe_load(open(mapping_yaml_path))
        # Build per-leg dict from pool names in mapping.yaml
        legs = {}
        adh = {}
        for spec in mp["actuators"]:
            pool = spec["pool"]
            # leg label: L1/R1/L2/R2/L3/R3 at start of pool name
            m = re.match(r'^(L[123]|R[123])_', pool)
            if m:
                leg = m.group(1)
                entry = legs.setdefault(leg, {})
                if "_coxa_ext" in pool:  entry["coxa_ext"]  = pool
                if "_coxa_flex" in pool: entry["coxa_flex"] = pool
                if "_femur_"   in pool:  entry["femur"]     = pool
                if "_tibia_"   in pool:  entry["tibia"]     = pool
            # adhesion pools: pool name starts with 'adh_' and actuator name encodes thorax/side
            if pool.startswith("adh_"):
                # map adh_adhere_claw_T1_left -> leg key 'L1', etc.
                if   "T1_left"  in pool: adh["L1"] = pool
                elif "T1_right" in pool: adh["R1"] = pool
                elif "T2_left"  in pool: adh["L2"] = pool
                elif "T2_right" in pool: adh["R2"] = pool
                elif "T3_left"  in pool: adh["L3"] = pool
                elif "T3_right" in pool: adh["R3"] = pool

        # attach adhesion to legs if present
        for leg, p in adh.items():
            legs.setdefault(leg, {})["adh"] = p

        self.legs = legs
        # advertise all pools so adapter validation is happy
        self._pools_known.update([v for d in legs.values() for v in d.values() if v])

    def _rect(self, x):  # half-wave rectifier 0..1
        return self.amp * max(0.0, math.sin(x))

    def readout(self, pool_names):
        out = {}
        t  = self._t_ms/1000.0
        w  = 2*math.pi*self.freq
        tripodA = {"L1","R2","L3"}

        for leg, chans in self.legs.items():
            base = 0.0 if leg in tripodA else math.pi     # A vs B
            # Joint phases
            phi_coxa_ext  = base + 0.0          # swing
            phi_coxa_flex = base + math.pi      # stance (antagonist)
            phi_femur     = base + math.pi/3    # lift a bit before swing
            phi_tibia     = base + math.pi/2    # lift more sharply

            # Rates (Hz); adapter low-pass handles smoothing
            if chans.get("coxa_ext"):
                out[chans["coxa_ext"]]  = {"mode":"rate","value": 140.0 * self._rect(w*t + phi_coxa_ext)}
            if chans.get("coxa_flex"):
                out[chans["coxa_flex"]] = {"mode":"rate","value": 140.0 * self._rect(w*t + phi_coxa_flex)}
            if chans.get("femur"):
                out[chans["femur"]]     = {"mode":"rate","value": 120.0 * self._rect(w*t + phi_femur)}
            if chans.get("tibia"):
                out[chans["tibia"]]     = {"mode":"rate","value": 120.0 * self._rect(w*t + phi_tibia)}

            # Adhesion: high in stance, low in swing (use coxa_flex phase)
            if chans.get("adh"):
                stance = self._rect(w*t + phi_coxa_flex)   # 0..1
                # Scale to a comfortable Hz range; adapter maps to force with logistic
                out[chans["adh"]] = {"mode":"rate","value": 200.0 * stance}

        # Ensure every asked-for pool exists
        for p in pool_names:
            out.setdefault(p, {"mode":"rate","value": 0.0})
        return out

# Export it so the adapter picks it up
bridge_mod.NeuronBridge = TripodCPGBridgeAF
print("TripodCPGBridgeAF installed ✅")


In [ ]:
# Point the bridge at YOUR mapping.yaml so it learns all pool names (incl. antagonists + adhesion)
from neuron_bridge.bridge import NeuronBridge   # now TripodCPGBridgeAF
import pathlib, yaml
mapping_yaml = str(pathlib.Path(NEURO_FLY) / "neuro_adapter" / "mapping.yaml")

# Use the antagonist-aware CPG we installed earlier
bridge = NeuronBridge(freq_hz=1.0, amp=0.6)   # was ~2.0/1.0; slower & smaller
bridge.set_mapping_from_yaml(str(Path(NEURO_FLY)/"neuro_adapter"/"mapping.yaml"))

import yaml, pathlib
mpath = pathlib.Path(NEURO_FLY)/"neuro_adapter"/"mapping.yaml"
mp = yaml.safe_load(open(mpath))

for spec in mp["actuators"]:
    # 1) lower torque gains
    if spec.get("type","") == "torque":
        if "coxa" in spec["actuator"]:
            spec["gain"] = 0.12   # was ~0.20–0.25
        elif "femur" in spec["actuator"] or "tibia" in spec["actuator"]:
            spec["gain"] = 0.10   # was ~0.15–0.18
        # 2) stronger activation smoothing (ms)
        spec["tau_decay_ms"] = 35.0    # was 10
        spec["tau_rise_ms"]  = 6.0     # was 2
        spec["kappa"]        = 0.03    # smaller input gain to the filter

    # 3) adhesion: softer or off
    if spec.get("type","") == "adhesion":
        spec["fmax"] = 1.2    # was 3-4
        spec["k"]    = 6.0    # gentler slope
        spec["x0"]   = 0.6    # sticks only when stance high

with open(mpath,"w") as f:
    yaml.safe_dump(mp,f,sort_keys=False)
print("mapping.yaml updated with calmer gains & filters ✅")


In [ ]:
from IPython.display import Video
Video("/path/to/local/digifly-legacy/neuro_fly/outputs/walk_tripod_full.mp4", embed=True, width=720)


In [ ]:
# === ONE BLOCK: no-ghost env + calm tripod gait with PD shaping → video ===
# - Disables the white "reference/trajectory ghost" (by hiding overlay after env creation)
# - Uses walk_imitation factory (no HDF5)
# - Auto-maps all legs (coxa/femur/tibia); adhesion OFF
# - Smooth CPG + local PD control
# - Writes & displays MP4

import os, sys, re, math, importlib, yaml, numpy as np
from pathlib import Path

# ---------- paths ----------
NEURO_FLY = globals().get("NEURO_FLY", "/path/to/local/digifly-legacy/neuro_fly")
if NEURO_FLY not in sys.path: sys.path.insert(0, NEURO_FLY)
mpath = Path(NEURO_FLY)/"neuro_adapter"/"mapping.yaml"

# ---------- util: hide reference/ghost overlay on an env ----------
def hide_reference_ghost(env, verbose=True):
    ph = env.physics
    # Make everything opaque (kills see-through that amplifies flicker)
    try:
        ph.model.geom_rgba[:, 3] = 1.0
    except Exception:
        pass
    # Hide geoms/sites/skins whose names look like overlays
    kill = ("ghost","ref","reference","traj","trajectory","mocap","target","marker","skeleton","spline")
    # geoms
    for i in range(ph.model.ngeom):
        name = (ph.model.id2name(i, 'geom') or "").lower()
        if any(k in name for k in kill):
            ph.model.geom_rgba[i, 3] = 0.0
    # sites
    for i in range(ph.model.nsite):
        name = (ph.model.id2name(i, 'site') or "").lower()
        if any(k in name for k in kill):
            ph.model.site_rgba[i, 3] = 0.0
    # skins (if present)
    try:
        for i in range(ph.model.nskin):
            name = (ph.model.id2name(i, 'skin') or "").lower()
            if any(k in name for k in kill):
                ph.model.skin_rgba[i, 3] = 0.0
    except Exception:
        pass
    if verbose: print("Reference overlay hidden.")

# ---------- patch env loader: NO GHOST (call factory, then hide overlay) ----------
import neuro_adapter.env_loader as el
import flybody.fly_envs as fb

def load_env_no_ghost(mode: str = "walk", **kwargs):
    m = mode.lower()
    if m in ("walk","walking","walk_imitation"):
        env = fb.walk_imitation(**kwargs)    # no unsupported kwargs here
        hide_reference_ghost(env, verbose=False)
        return env
    if m in ("walk_on_ball","ball"):
        env = fb.walk_on_ball(**kwargs)
        hide_reference_ghost(env, verbose=False)
        return env
    if m in ("flight","fly","flight_wpg"):
        env = fb.flight_imitation(**kwargs)
        hide_reference_ghost(env, verbose=False)
        return env
    if hasattr(fb, "make_env"):
        env = fb.make_env(mode, **kwargs)
        hide_reference_ghost(env, verbose=False)
        return env
    raise ValueError(f"Unknown mode '{mode}'")

el.load_env = load_env_no_ghost

# Rebind anything that imported the old loader
import neuro_adapter.adapter as adapter_mod
importlib.reload(adapter_mod)
from neuro_adapter.env_loader import load_env, list_actuator_names

# ---------- build calm mapping from the actual env (no adhesion) ----------
env = load_env("walk")  # WalkImitation, overlay disabled by loader
names = list_actuator_names(env)

def find_name(substr, name_list):
    for n in name_list:
        if substr in n:
            return n
    return None

legs = [("L1","T1","left"), ("R1","T1","right"),
        ("L2","T2","left"), ("R2","T2","right"),
        ("L3","T3","left"), ("R3","T3","right")]

actuators = []
for label, thorax, side in legs:
    coxa  = find_name(f"walker/coxa_{thorax}_{side}", names)
    femur = find_name(f"walker/femur_{thorax}_{side}", names)
    tibia = find_name(f"walker/tibia_{thorax}_{side}", names)
    if coxa:
        actuators += [
            {"pool": f"{label}_coxa_ext",  "actuator": coxa,  "type":"torque",
             "gain":0.10, "sign":+1.0, "tau_decay_ms":50.0, "tau_rise_ms":10.0, "kappa":0.02},
            {"pool": f"{label}_coxa_flex", "actuator": coxa,  "type":"torque",
             "gain":0.10, "sign":-1.0, "tau_decay_ms":50.0, "tau_rise_ms":10.0, "kappa":0.02},
        ]
    if femur:
        actuators.append({"pool": f"{label}_femur_drive","actuator": femur,"type":"torque",
                          "gain":0.08, "sign":+1.0, "tau_decay_ms":50.0, "tau_rise_ms":10.0, "kappa":0.02})
    if tibia:
        actuators.append({"pool": f"{label}_tibia_drive","actuator": tibia,"type":"torque",
                          "gain":0.08, "sign":+1.0, "tau_decay_ms":50.0, "tau_rise_ms":10.0, "kappa":0.02})

mapping = {
    "global": {
        "neural_dt_ms": 0.025,
        "clip_actions": True,
        "action_smoothing_tau_ms": 60.0
    },
    "actuators": actuators
}
with open(mpath,"w") as f: yaml.safe_dump(mapping, f, sort_keys=False)
print(f"[mapping] wrote {mpath} with {len(actuators)} entries")

# ---------- smooth tripod CPG ----------
from neuron_bridge import bridge as bridge_mod

class SmoothTripodCPG(bridge_mod.DummyBridge):
    def __init__(self, freq_hz=0.5, amp=0.45):
        super().__init__({})
        self.freq, self.amp = float(freq_hz), float(amp)
        self.legs = {}; self._pools_known=set()
    def set_mapping_from_yaml(self, mapping_yaml_path):
        mp = yaml.safe_load(open(mapping_yaml_path))
        legs={}
        for spec in mp["actuators"]:
            p = spec["pool"]
            for key in ("coxa_ext","coxa_flex","femur_drive","tibia_drive"):
                if f"_{key}" in p:
                    leg = p.split(f"_{key}")[0]
                    legs.setdefault(leg, {})[key] = p
        self.legs=legs
        self._pools_known.update([v for d in legs.values() for v in d.values()])
    def readout(self, pool_names):
        out={}; t=self._t_ms/1000.0; w=2*math.pi*self.freq; A={"L1","R2","L3"}
        for leg,ch in self.legs.items():
            base = 0.0 if leg in A else math.pi
            s   = 0.5*(1+math.sin(w*t + base)) * self.amp
            sm  = 0.5*(1+math.sin(w*t + base + math.pi)) * self.amp
            sf  = 0.5*(1+math.sin(w*t + base + math.pi/3)) * self.amp
            st  = 0.5*(1+math.sin(w*t + base + math.pi/2)) * self.amp
            if "coxa_ext"    in ch: out[ch["coxa_ext"]]    = {"mode":"rate","value":100.0*s}
            if "coxa_flex"   in ch: out[ch["coxa_flex"]]   = {"mode":"rate","value":100.0*sm}
            if "femur_drive" in ch: out[ch["femur_drive"]] = {"mode":"rate","value":90.0*sf}
            if "tibia_drive" in ch: out[ch["tibia_drive"]] = {"mode":"rate","value":90.0*st}
        for p in pool_names:
            out.setdefault(p, {"mode":"rate","value":0.0})
        return out

bridge = SmoothTripodCPG()
bridge.set_mapping_from_yaml(str(mpath))
bridge_mod.NeuronBridge = SmoothTripodCPG  # so other code paths pick it up too

# ---------- PD-shaped adapter ----------
from neuro_adapter.adapter import NeuroMuJoCoAdapter as _Base

class PDAdapter(_Base):
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.kp = {"coxa": 2.5, "femur": 2.2, "tibia": 2.0}
        self.kd = {"coxa": 0.06, "femur": 0.05, "tibia": 0.04}
        self.amp= {"coxa": math.radians(6.0), "femur": math.radians(7.0), "tibia": math.radians(6.0)}
        ph = self.env.physics
        self._act2joint={}
        for name, idx in self.actu_index.items():
            try:
                j_id = int(ph.model.actuator_trnid[idx,0])
                if j_id < 0: continue
                qpos_adr = int(ph.model.jnt_qposadr[j_id])
                qvel_adr = int(ph.model.jnt_dofadr[j_id])
                if "coxa_" in name: k="coxa"
                elif "femur_" in name: k="femur"
                elif "tibia_" in name: k="tibia"
                else: k=None
                self._act2joint[idx]=(j_id,qpos_adr,qvel_adr,k)
            except Exception:
                pass
        try:
            ph.model.opt.timestep = max(ph.model.opt.timestep, 0.001)
            ph.model.dof_damping[:] = ph.model.dof_damping[:] * 1.2
        except Exception:
            pass

    def run(self, *a, **kw):
        import numpy as np
        ts = self.env.reset()
        self.bridge.set_stim_plan(kw.get("stim_plan", {}))
        t_ms=0.0; end_ms=float(kw.get("duration_s",6.0))*1000.0
        cdt=self.control_dt*1000.0; ndt=self.neural_dt*1000.0
        last=0.0; period=(1000.0/self.render_fps) if self.vw else None
        cam = kw.get("render_cam", None)
        tau_ms = float(self.mapping.get("global",{}).get("action_smoothing_tau_ms",60.0))
        u_prev=None

        while t_ms < end_ms:
            self.bridge.window_reset()
            for _ in range(self.steps_per_action):
                self.bridge.step(ndt); t_ms += ndt

            pools=[s["pool"] for s in self.mapping.get("actuators",[])]
            ro=self.bridge.readout(pools)
            activ={}
            for s in self.mapping.get("actuators", []):
                pool=s["pool"]; flt=self._filters[pool]
                r=ro.get(pool, {"mode":"rate","value":0.0})
                if r.get("mode")=="spikes":
                    a_val = flt.update_from_spikes(r.get("times_ms",[]), t_ms, t_ms-cdt)
                else:
                    a_val = flt.update_from_rate(float(r.get("value",0.0)), cdt, r_max=s.get("r_max",200.0))
                activ[pool]=a_val

            ph = self.env.physics
            u = np.zeros(self.action_size, dtype=np.float64)

            for spec in self.mapping.get("actuators", []):
                name=spec["actuator"]; idx=self.actu_index[name]
                meta=self._act2joint.get(idx); 
                if not meta: continue
                j_id,qpos_adr,qvel_adr,kind = meta

                if "_coxa_" in name:
                    legL = "L" if "left" in name else ("R" if "right" in name else None)
                    seg  = "1" if "_T1_" in name else ("2" if "_T2_" in name else "3")
                    leg  = f"{legL}{seg}" if legL else None
                    if leg:
                        a_ext = activ.get(f"{leg}_coxa_ext",0.0)
                        a_flex= activ.get(f"{leg}_coxa_flex",0.0)
                        a_goal = (a_ext - a_flex) * self.amp["coxa"]
                    else:
                        a_goal = 0.0
                elif "_femur_" in name:
                    leg = ("L" if "left" in name else "R") + ("1" if "_T1_" in name else ("2" if "_T2_" in name else "3"))
                    a_goal = activ.get(f"{leg}_femur_drive",0.0) * self.amp["femur"]
                elif "_tibia_" in name:
                    leg = ("L" if "left" in name else "R") + ("1" if "_T1_" in name else ("2" if "_T2_" in name else "3"))
                    a_goal = activ.get(f"{leg}_tibia_drive",0.0) * self.amp["tibia"]
                else:
                    a_goal = 0.0

                q  = float(ph.data.qpos[qpos_adr])
                qd = float(ph.data.qvel[qvel_adr])
                kind = kind or ("coxa" if "coxa" in name else ("femur" if "femur" in name else "tibia"))
                kp = self.kp.get(kind,2.0); kd=self.kd.get(kind,0.05)
                tau = kp*(a_goal - q) + kd*(0.0 - qd)
                u[idx] += tau

            # global action smoothing
            if tau_ms > 0:
                if u_prev is None: u_prev = u.copy()
                alpha = min(1.0, cdt/tau_ms)
                u = u_prev + alpha*(u - u_prev)
                u_prev = u

            # clamp & step
            spec_act = self.env.action_spec()
            u = np.clip(u, spec_act.minimum, spec_act.maximum)
            ts = self.env.step(u)

            if self.vw is not None and ((period is None) or ((t_ms-last) >= period)):
                try:
                    H,W = self.render_size
                    frame=self.env.physics.render(height=H, width=W, camera_id=cam)
                    self.vw.add(frame); last=t_ms
                except Exception as e:
                    print(f"[WARN] Render failed: {e}")
        if self.vw is not None: self.vw.close()

# use our PD adapter everywhere
adapter_mod.NeuroMuJoCoAdapter = PDAdapter
importlib.reload(adapter_mod)
from neuro_adapter.adapter import NeuroMuJoCoAdapter

# ---------- run one episode ----------
video_path = str(Path(NEURO_FLY)/"outputs"/"walk_pd_no_ghost.mp4")
adapter = NeuroMuJoCoAdapter(
    mode="walk",
    mapping=yaml.safe_load(open(mpath)),
    bridge=bridge,
    render_fps=30,
    video_path=video_path,
    render_size=(720,720),
)

# tighten FOV
def set_fov(env, cam_name, fovy_deg=20.0):
    ph = env.physics
    idx = [i for i in range(ph.model.ncam) if ph.model.id2name(i,'camera')==cam_name][0]
    ph.model.cam_fovy[idx] = float(fovy_deg)

cam = "walker/hero"
set_fov(adapter.env, cam, 20.0)

adapter.run(stim_plan={"stimuli":[]}, duration_s=8.0, render_cam=cam)
print("Video:", video_path)

from IPython.display import Video as _V
_V(video_path, embed=True, width=720)


In [ ]:


# === EXACT INTROSPECTION + REFERENCE TRACKING (no guessing) ===
import os, sys, math, inspect, re, textwrap
from pathlib import Path
import numpy as np, h5py

import flybody.fly_envs as fb

# ---------- CONFIG ----------
HDF5 = "/path/to/local/downloads/25309105/datasets_walking-imitation/walking-dataset_female-only_snippets-16252_trk-files-0-9.hdf5"
TRAJ_INDEX = 0              # change to any valid trajectory index
VIDEO_OUT  = "./walk_PD_track.mp4"
RENDER_FPS = 30
DURATION_S = 6.0

assert Path(HDF5).exists(), f"Missing HDF5: {HDF5}"

LINE = "─"*96
def hdr(s): print("\n"+LINE+"\n"+s+"\n"+LINE)

# ---------- 1) Show exact signatures ----------
Loader = fb.HDF5WalkingTrajectoryLoader
walk_factory = fb.walk_imitation
print("Loader.__init__ =", inspect.signature(Loader.__init__))
print("walk_imitation  =", inspect.signature(walk_factory))

# ---------- 2) Build loader using the documented API ----------
loader = Loader(path=HDF5)   # your build: (path, traj_indices=None, random_state=None)
print("num_trajectories:", loader.num_trajectories)
print("timestep (s):    ", float(loader.timestep))

# ---------- 3) Query official methods (no schema guessing) ----------
joint_names = loader.get_joint_names()
site_names  = loader.get_site_names()
T_len       = loader.trajectory_len(TRAJ_INDEX)
traj_obj    = loader.get_trajectory(TRAJ_INDEX)

hdr("LOADER REPORT")
print(f"- get_joint_names(): {len(joint_names)} names. Sample: {joint_names[:8]}")
print(f"- get_site_names():  {len(site_names)} names. Sample: {site_names[:5]}")
print(f"- trajectory_len({TRAJ_INDEX}): {T_len} frames")
print(f"- get_trajectory({TRAJ_INDEX}) type: {type(traj_obj).__name__}")

# Inspect the trajectory object deterministically
def describe_traj_obj(traj):
    items = []
    if isinstance(traj, dict):
        it = traj.items()
    else:
        # collect public attrs
        it = [(n, getattr(traj, n)) for n in dir(traj) if not n.startswith("_")]
    for k,v in it:
        try:
            if isinstance(v, (np.ndarray, h5py.Dataset)):
                shape = tuple(v.shape)
                kind  = "ndarray" if isinstance(v, np.ndarray) else "h5py.Dataset"
                items.append((k, kind, shape, str(v.dtype)))
            elif isinstance(v, (list, tuple)):
                items.append((k, type(v).__name__, (len(v),), type(v[0]).__name__ if v else "empty"))
            elif isinstance(v, (int,float,str,bool)):
                items.append((k, type(v).__name__, (), repr(v)))
        except Exception:
            pass
    return items

desc = describe_traj_obj(traj_obj)
print("Trajectory members (arrays/lists/scalars):")
for k,typ,shape,d in sorted(desc):
    print(f"  - {k:24s} {typ:14s} shape={shape} dtype/info={d}")

# Choose the joint-angle array: prefer (T x N) whose N == len(joint_names)
def pick_angles(traj, nj):
    candidates = []
    # helper to iterate kv pairs
    if isinstance(traj, dict):
        it = traj.items()
    else:
        it = [(n, getattr(traj,n)) for n in dir(traj) if not n.startswith("_")]
    for k,v in it:
        if isinstance(v, (np.ndarray, h5py.Dataset)) and getattr(v,'ndim',0)==2:
            T,N = v.shape
            score = 0
            if N == nj: score = 100
            elif abs(N - nj) <= max(1, int(0.1*nj)): score = 50  # near match
            # keyword hints
            lk = k.lower()
            if any(t in lk for t in ("qpos","angle","joint","pose","theta")): score += 10
            if score>0: candidates.append((score, k, v))
    if not candidates:
        return None, None
    candidates.sort(reverse=True, key=lambda x: x[0])
    best = candidates[0]
    return best[1], best[2]

angle_key, angle_array = pick_angles(traj_obj, len(joint_names))
if angle_array is None:
    raise RuntimeError("Could not find a (T x N) joint-angle array in the trajectory object. "
                       "See the printed members above and tell me the correct key.")

print(f"\nChosen joint-angle key: '{angle_key}' with shape={tuple(angle_array.shape)} "
      f"(expect N≈{len(joint_names)})")

# ---------- 4) Build env via factory (uses the HDF5 internally) ----------
# We'll disable early termination while we render/track, and keep default joint_filter.
env = walk_factory(ref_path=HDF5, traj_indices=[TRAJ_INDEX], terminal_com_dist=float('inf'))
ph = env.physics

# Hide the white reference/ghost overlay (so you see only the physical fly)
def hide_reference_ghost(env):
    ph = env.physics
    try: ph.model.geom_rgba[:,3] = 1.0
    except Exception: pass
    kill = ("ghost","ref","reference","traj","trajectory","mocap","target","marker","skeleton","spline")
    for i in range(ph.model.ngeom):
        name = (ph.model.id2name(i,'geom') or "").lower()
        if any(k in name for k in kill): ph.model.geom_rgba[i,3] = 0.0
    for i in range(ph.model.nsite):
        name = (ph.model.id2name(i,'site') or "").lower()
        if any(k in name for k in kill): ph.model.site_rgba[i,3] = 0.0
    try:
        for i in range(ph.model.nskin):
            name = (ph.model.id2name(i,'skin') or "").lower()
            if any(k in name for k in kill): ph.model.skin_rgba[i,3] = 0.0
    except Exception: pass

hide_reference_ghost(env)

# ---------- 5) Map dataset joints -> env joints ----------
env_joint_names = [ph.model.id2name(i,'joint') for i in range(ph.model.njnt)]
def norm(n:str)->str:
    return (n or "").lower().replace("walker/","").replace("_axis","").replace(":0","")
env_name2qpos = {norm(n): int(ph.model.jnt_qposadr[i]) for i,n in enumerate(env_joint_names)}

ref2env_qpos = {}
for j, rname in enumerate(joint_names):
    rn = norm(rname)
    # exact
    target = env_name2qpos.get(rn)
    # relaxed (token containment)
    if target is None:
        for en,qp in env_name2qpos.items():
            if rn in en or en in rn:
                target = qp; break
    ref2env_qpos[j] = target

matched = sum(1 for v in ref2env_qpos.values() if v is not None)
hdr("MAPPING REPORT")
print(f"Env joints: {len(env_joint_names)}")
print(f"Dataset joints: {len(joint_names)}")
print(f"Matched dataset→env qpos addresses: {matched}/{len(joint_names)}")

if matched < max(6, int(0.5*len(joint_names))):
    # show some unmatched for troubleshooting
    missing = [joint_names[j] for j,v in ref2env_qpos.items() if v is None][:20]
    print("WARNING: many joints unmatched. First few missing names:", missing)

# ---------- 6) Build actuator -> (qpos, qvel, name) map ----------
act2joint = {}
for a in range(ph.model.nu):
    j_id = int(ph.model.actuator_trnid[a,0])
    if j_id >= 0:
        act2joint[a] = (
            int(ph.model.jnt_qposadr[j_id]),
            int(ph.model.jnt_dofadr[j_id]),
            ph.model.id2name(j_id, 'joint')
        )

# ---------- 7) Minimal PD tracker so the PHYSICAL body follows the dataset ----------
def gains_for_joint(name):
    n = (name or "").lower()
    if "coxa"  in n: return 3.0, 0.06
    if "femur" in n: return 2.5, 0.05
    if "tibia" in n: return 2.0, 0.04
    return 2.5, 0.05

# Camera framing
def set_fov(env, cam="walker/hero", fovy_deg=20.0):
    idx = [i for i in range(env.physics.model.ncam) if env.physics.model.id2name(i,'camera')==cam][0]
    env.physics.model.cam_fovy[idx] = float(fovy_deg)
set_fov(env, "walker/hero", 20.0)

# Video writer (use repo helper; fall back to imageio if needed)
try:
    from neuro_adapter.video import VideoWriter
    vw = VideoWriter(VIDEO_OUT, fps=RENDER_FPS)
    _use_imageio = False
except Exception:
    import imageio.v3 as iio
    writer = iio.get_writer(VIDEO_OUT, fps=RENDER_FPS, codec="libx264", quality=7)
    _use_imageio = True
    class _VW:
        def add(self, frame): writer.append_data(frame)
        def close(self): writer.close()
    vw = _VW()

# Time bases
ctrl_dt = float(env.control_timestep())
ref_dt  = float(loader.timestep)   # exact from loader
T, N    = angle_array.shape

def ref_index_at_time(t):
    i = int(round(t / ref_dt))
    return 0 if i < 0 else (T-1 if i >= T else i)

# Rollout
ts = env.reset()
t = 0.0
ctrl_dt = float(env.control_timestep())
steps = int(DURATION_S / ctrl_dt)

for _ in range(steps):
    # Always refresh the physics handle AFTER each step
    ph = env.physics

    # sample one reference frame (cheap even for 6GB HDF5)
    ridx = ref_index_at_time(t)
    ref_frame = np.asarray(angle_array[ridx]) if hasattr(angle_array, "dtype") else angle_array[ridx]

    # build torques
    u = np.zeros(env.action_spec().shape[0], dtype=np.float64)
    for a_idx, (qpos_adr, qvel_adr, jname) in act2joint.items():
        # find the ref column mapped to this env qpos
        target = None
        for jcol, env_qpos_idx in ref2env_qpos.items():
            if env_qpos_idx == qpos_adr:
                target = float(ref_frame[jcol]); break
        if target is None:
            continue

        # IMPORTANT: read q, qd from the *fresh* physics object
        try:
            q  = float(ph.data.qpos[qpos_adr])
            qd = float(ph.data.qvel[qvel_adr])
        except ReferenceError:
            # if a swap just happened mid-iteration, refresh and retry once
            ph = env.physics
            q  = float(ph.data.qpos[qpos_adr])
            qd = float(ph.data.qvel[qvel_adr])

        kp, kd = gains_for_joint(jname)
        u[a_idx] = kp*(target - q) + kd*(0.0 - qd)

    # clamp & step
    spec = env.action_spec()
    u = np.clip(u, spec.minimum, spec.maximum)
    ts = env.step(u)

    # render using a fresh physics handle too
    frame = env.physics.render(height=H, width=W, camera_id=cam)
    vw.add(frame)

    t += ctrl_dt


vw.close()
print(f"\nWROTE VIDEO: {Path(VIDEO_OUT).resolve()}")

# Inline playback (Jupyter)
from IPython.display import Video
Video(VIDEO_OUT, embed=True, width=720)




In [ ]:
# === FIXED: use qpos offset, loop snippet, slow playback, refresh physics each step ===
import numpy as np
from pathlib import Path
import flybody.fly_envs as fb

# Assumes you've already set:
#   HDF5        -> your 6GB file path
#   loader      -> fb.HDF5WalkingTrajectoryLoader(path=HDF5)
#   joint_names -> loader.get_joint_names()
#   traj_obj    -> loader.get_trajectory(TRAJ_INDEX)
#   angle_array -> traj_obj["qpos"]     (shape (T, 109))
# If not, uncomment:
# loader = fb.HDF5WalkingTrajectoryLoader(path=HDF5)
# joint_names = loader.get_joint_names()
# traj_obj = loader.get_trajectory(0)
# angle_array = traj_obj["qpos"]

# 1) Compute and assert the qpos offset (root: x,y,z, quat[4] = 7)
T, N = angle_array.shape
nj = len(joint_names)
offset = N - nj
assert offset >= 0, f"Bad qpos/joint sizes: N={N}, nj={nj}"
print(f"[OK] qpos columns = {N}, joints = {nj}, using OFFSET = {offset}")

# 2) Build env from factory with your chosen trajectory index; keep it alive while rendering
TRAJ_INDEX = 0
env = fb.walk_imitation(ref_path=HDF5, traj_indices=[TRAJ_INDEX],
                        terminal_com_dist=float('inf'), joint_filter=0.02)

# Hide the ghost/reference overlay (so you only see the physical body)
def hide_reference_ghost(env):
    ph = env.physics
    try: ph.model.geom_rgba[:,3] = 1.0
    except Exception: pass
    kill = ("ghost","ref","reference","traj","trajectory","mocap","target","marker","skeleton","spline")
    for i in range(ph.model.ngeom):
        nm = (ph.model.id2name(i,'geom') or "").lower()
        if any(k in nm for k in kill): ph.model.geom_rgba[i,3] = 0.0
    for i in range(ph.model.nsite):
        nm = (ph.model.id2name(i,'site') or "").lower()
        if any(k in nm for k in kill): ph.model.site_rgba[i,3] = 0.0
    try:
        for i in range(ph.model.nskin):
            nm = (ph.model.id2name(i,'skin') or "").lower()
            if any(k in nm for k in kill): ph.model.skin_rgba[i,3] = 0.0
    except Exception: pass
hide_reference_ghost(env)

# 3) Map dataset joints -> env joints (by name)
ph_model = env.physics.model
env_joint_names = [ph_model.id2name(i,'joint') for i in range(ph_model.njnt)]
def norm(n): return (n or "").lower().replace("walker/","").replace("_axis","").replace(":0","")
env_name2qpos = {norm(n): int(ph_model.jnt_qposadr[i]) for i,n in enumerate(env_joint_names)}
ref2env_qpos = {}
for j, rname in enumerate(joint_names):
    rn = norm(rname)
    target = env_name2qpos.get(rn)
    if target is None:
        for en, qp in env_name2qpos.items():
            if rn in en or en in rn:
                target = qp; break
    ref2env_qpos[j] = target
matched = sum(1 for v in ref2env_qpos.values() if v is not None)
print(f"[MAP] matched {matched}/{len(joint_names)} dataset joints to env joints")

# 4) Actuator -> (qpos, qvel, name)
act2joint = {}
for a in range(ph_model.nu):
    j_id = int(ph_model.actuator_trnid[a,0])
    if j_id >= 0:
        act2joint[a] = (
            int(ph_model.jnt_qposadr[j_id]),
            int(ph_model.jnt_dofadr[j_id]),
            ph_model.id2name(j_id, 'joint')
        )

# 5) PD gains per joint family (tame)
def gains_for_joint(name):
    n = (name or "").lower()
    if "coxa"  in n: return 3.0, 0.06
    if "femur" in n: return 2.5, 0.05
    if "tibia" in n: return 2.0, 0.04
    return 2.5, 0.05

# Slightly increase damping for stability
try:
    ph_model.dof_damping[:] = ph_model.dof_damping[:] * 1.4
except Exception:
    pass

# 6) Playback timing: LOOP and SLOW DOWN (snippets are only 90 frames at dt=0.002s = 0.18s)
ctrl_dt = float(env.control_timestep())
ref_dt  = float(loader.timestep)   # 0.002 from your report
playback_speed = 0.35              # <1 slows it down; try 0.25–0.5
period = T * ref_dt / playback_speed

def ref_index_at_time(t):
    # loop the snippet seamlessly at slower playback_speed
    t_mod = t % period
    i = int((t_mod * playback_speed) / ref_dt) % T
    return i

# 7) Run & render
from neuro_adapter.video import VideoWriter
out = Path("./walk_PD_track_offset_fixed.mp4").resolve()
vw = VideoWriter(str(out), fps=30)
H, W, cam = 720, 720, "walker/hero"

ts = env.reset()
t  = 0.0
steps = int(6.0 / ctrl_dt)

for _ in range(steps):
    ph = env.physics  # IMPORTANT: refresh physics each step (avoid weakref issues)

    ridx = ref_index_at_time(t)
    ref_frame = np.asarray(angle_array[ridx])  # length N (= 109)

    # Build torques with CORRECT qpos offset
    u = np.zeros(env.action_spec().shape[0], dtype=np.float64)
    for a_idx, (qpos_adr, qvel_adr, jname) in act2joint.items():
        # find which dataset joint maps to this qpos_adr
        target = None
        for jcol, env_qpos_idx in ref2env_qpos.items():
            if env_qpos_idx == qpos_adr:
                target = float(ref_frame[offset + jcol])  # <<< OFFSET APPLIED HERE
                break
        if target is None:
            continue
        q  = float(ph.data.qpos[qpos_adr])
        qd = float(ph.data.qvel[qvel_adr])
        kp, kd = gains_for_joint(jname)
        u[a_idx] = kp*(target - q) + kd*(0.0 - qd)

    spec = env.action_spec()
    u = np.clip(u, spec.minimum, spec.maximum)
    ts = env.step(u)

    frame = env.physics.render(height=H, width=W, camera_id=cam)
    vw.add(frame)

    t += ctrl_dt

vw.close()
print("Wrote:", out)

from IPython.display import Video
Video(str(out), embed=True, width=720)


In [ ]:
%%bash
set -euxo pipefail

# Use classic solver (avoids the libmambapy error)
conda config --set solver classic

# Create a Python 3.10 env compatible with tensorflow-macos
conda create -n flywalk-py310 python=3.10 -y

# Install Apple-Silicon TF + deps
conda run -n flywalk-py310 pip install \
  "tensorflow-macos==2.15.0" "tensorflow-metal==1.1.0" \
  mujoco==3.1.6 dm-control==1.0.14 \
  acme==0.4.0 \
  imageio[ffmpeg] matplotlib absl-py

# Install your local FlyBody (edit the path if your repo lives elsewhere)
conda run -n flywalk-py310 pip install -e "/path/to/local/flybody-main"

# Register the env as a Jupyter kernel
conda run -n flywalk-py310 python -m ipykernel install --user \
  --name flywalk-py310 --display-name "Python (flywalk-py310)"


In [ ]:
import tensorflow as tf
from acme import wrappers
from flybody.fly_envs import walk_imitation
from flybody.tasks.task_utils import get_random_policy, real2canonical
from flybody.agents.utils_tf import TestPolicyWrapper
from flybody.utils import rollout_and_render, display_video

# 1) set your paths
ref_walking_path = "/path/to/local/downloads/25309105/datasets_walking-imitation/walking-dataset_female-only_snippets-16252_trk-files-0-9.hdf5"
walk_policy_path = "/path/to/local/downloads/25309105/trained-fly-policies/walking"   # <- folder with SavedModel from the download

# 2) make env
env = walk_imitation(ref_path=ref_walking_path, terminal_com_dist=float('inf'))
env = wrappers.SinglePrecisionWrapper(env)
env = wrappers.CanonicalSpecWrapper(env, clip=True)

# (optional) pick a longer trajectory snippet the authors used in the notebook
env.task.set_next_trajectory_index(idx=316)

# 3) load pretrained policy
walking_policy = tf.saved_model.load(walk_policy_path)
walking_policy = TestPolicyWrapper(walking_policy)

# 4) roll out and render
frames = rollout_and_render(env, walking_policy, run_until_termination=True, camera_ids=2, height=720, width=720)
display_video(frames)


In [ ]:
# === INTROSPECT EVERYTHING: loader signatures, HDF5 tree, candidates for names/angles/fps ===
import os, sys, inspect, textwrap, re
from pathlib import Path
import numpy as np

HDF5 = "/path/to/local/downloads/25309105/datasets_walking-imitation/walking-dataset_female-only_snippets-16252_trk-files-0-9.hdf5"  # <-- set this
assert Path(HDF5).exists(), f"Missing HDF5 at {HDF5}"

# ---- Imports ----
import h5py
import flybody.fly_envs as fb

LINE = "─"*88

def header(t):
    print("\n" + LINE)
    print(t)
    print(LINE)

def short(s, n=300):
    s = str(s)
    return (s[:n] + "…") if len(s) > n else s

# ---------- 1) Show signatures for loader + factory ----------
header("Signatures")
Loader = fb.HDF5WalkingTrajectoryLoader
walk_factory = fb.walk_imitation
walk_class = getattr(fb, "WalkImitation", None)

print("HDF5WalkingTrajectoryLoader.__init__ =", inspect.signature(Loader.__init__))
print("walk_imitation =", inspect.signature(walk_factory))
if walk_class:
    print("WalkImitation.__init__ =", inspect.signature(walk_class.__init__))

# (optional) first lines of docstrings
for name, obj in [("Loader.__init__", Loader.__init__), ("walk_imitation", walk_factory)]:
    doc = (inspect.getdoc(obj) or "").splitlines()
    if doc:
        print(f"\n{name} doc:", short(doc[0]))

# ---------- 2) Try to construct the loader (show exactly what works/fails) ----------
header("Instantiate HDF5WalkingTrajectoryLoader (report all attempts)")

lsig = inspect.signature(Loader.__init__)
params = [p for p in lsig.parameters if p != "self"]
print("Loader params:", params)

attempts = []
# Start with positional (HDF5 as first arg), then common kw names present in this build
kw_candidates = ["path","h5_path","hdf5_path","file","filename","dataset_path","source","fname","data_path"]
present = [k for k in kw_candidates if k in params]

attempts.append(((HDF5,), {}))
for k in present:
    attempts.append(((), {k: HDF5}))

# Add benign extras if they exist and have defaults (avoid forcing)
defaults = {}
for opt in ["split","partition","subset","sequence","trial","index","idx"]:
    if opt in params:
        # don't assume meaning; set a safe default only if the param has a default
        if lsig.parameters[opt].default is not inspect._empty:
            defaults[opt] = lsig.parameters[opt].default

loader = None
errors = []
for args, kwargs in attempts:
    # merge defaults carefully
    merged = dict(kwargs)
    merged.update({k:v for k,v in defaults.items() if k not in merged})
    try:
        loader = Loader(*args, **merged)
        print(f"✅ SUCCESS with args={args} kwargs={merged}")
        break
    except Exception as e:
        errors.append((args, merged, repr(e)))
        print(f"❌ FAIL   with args={args} kwargs={merged} -> {e}")

if loader is None:
    raise RuntimeError("Could not construct HDF5WalkingTrajectoryLoader. See failures above.")

# ---------- 3) Introspect the loader object: find names/angles/fps/dt ----------
header("Loader attribute sweep (shallow recursion)")

def as_list_of_str(x):
    try:
        if isinstance(x, (list, tuple)):
            return [s.decode("utf-8") if hasattr(s,"decode") else str(s) for s in x]
        if isinstance(x, np.ndarray) and x.dtype.kind in "OUS":
            return [s.decode("utf-8") if hasattr(s,"decode") else str(s) for s in x.tolist()]
    except Exception:
        pass
    return None

def iter_attrs(obj, max_depth=2, prefix="traj"):
    seen = set()
    def walk(o, depth, path):
        if depth > max_depth: return
        for name in dir(o):
            if name.startswith("_"): 
                continue
            key = (id(o), name)
            if key in seen: 
                continue
            seen.add(key)
            try:
                val = getattr(o, name)
            except Exception:
                continue
            tp = type(val).__name__
            if isinstance(val, np.ndarray):
                print(f"[{path}.{name}] ndarray shape={val.shape} dtype={val.dtype}")
            elif isinstance(val, h5py.Dataset):
                print(f"[{path}.{name}] h5py.Dataset shape={val.shape} dtype={val.dtype} path=/{val.name}")
            elif isinstance(val, (list, tuple)):
                print(f"[{path}.{name}] {tp} len={len(val)} sample={short(val[:5])}")
            elif isinstance(val, (str, bytes, int, float, bool)):
                print(f"[{path}.{name}] {tp} = {short(val)}")
            else:
                print(f"[{path}.{name}] {tp}")
            # Recurse into simple objects
            if hasattr(val, "__dict__") and not isinstance(val, (np.ndarray, h5py.Dataset, str, bytes)):
                walk(val, depth+1, f"{path}.{name}")
    walk(obj, 0, prefix)

iter_attrs(loader)

# Heuristics to extract likely candidates
header("Candidates inside loader (heuristics)")

names_cand = []
angles_cand = []
fps_cand, dt_cand = [], []

def collect_candidates(obj, max_depth=2):
    seen = set()
    def walk(o, depth):
        if depth > max_depth: return
        for name in dir(o):
            if name.startswith("_"): continue
            key = (id(o), name)
            if key in seen: continue
            seen.add(key)
            try:
                val = getattr(o, name)
            except Exception:
                continue
            low = name.lower()
            # names
            lst = as_list_of_str(val)
            if lst and 3 <= len(lst) <= 1000:
                if any(tok in low for tok in ("name","joint","mocap")):
                    names_cand.append((f"{name}", lst[:10], len(lst)))
            # angles/poses (2D arrays)
            if isinstance(val, (np.ndarray, h5py.Dataset)) and getattr(val, "ndim", 0) == 2:
                T,N = val.shape
                if T >= 10 and 1 <= N <= 5000:   # broad, safe bounds
                    if any(tok in low for tok in ("qpos","angle","pose","joint","mocap")):
                        angles_cand.append((f"{name}", val.shape, type(val).__name__))
            # timebase
            if low in ("fps","mocap_fps","sample_rate","rate"):
                try: fps_cand.append((name, float(val)))
                except: pass
            if low in ("dt","timestep","delta_t"):
                try: dt_cand.append((name, float(val)))
                except: pass
            if hasattr(val, "__dict__") and not isinstance(val, (np.ndarray, h5py.Dataset, str, bytes)):
                walk(val, depth+1)
    walk(obj, 0)

collect_candidates(loader)

print("Name arrays (first few items shown):")
for nm, sample, L in names_cand[:10]:
    print(f" - {nm}: len={L} sample={sample}")

print("\nAngle/pose arrays (2D):")
for nm, shape, typ in angles_cand[:15]:
    print(f" - {nm}: shape={shape} type={typ}")

print("\nTimebase candidates:")
print(" - fps:", fps_cand or "[]")
print(" - dt :", dt_cand or "[]")

# ---------- 4) Scan the HDF5 file tree (safe: sizes only, tiny samples) ----------
header("HDF5 tree (filtered candidates first, then top-level groups)")

def sample_dataset(dset, max_items=12):
    try:
        if dset.ndim == 1:
            n = min(max_items, dset.shape[0])
            arr = dset[:n]
            if arr.dtype.kind in "OUS":
                arr = [a.decode("utf-8") if hasattr(a,"decode") else str(a) for a in arr]
            else:
                arr = arr.tolist()
            return arr
        if dset.ndim == 2:
            r = dset[0:1, :min(max_items, dset.shape[1])]
            return np.asarray(r).tolist()
    except Exception as e:
        return f"<sample error: {e}>"
    return "<no sample>"

cands = []
with h5py.File(HDF5, "r") as f:
    # pass 1: collect likely name/angles/fps/dt datasets
    def visit(name, obj):
        if isinstance(obj, h5py.Dataset):
            nm = name.lower()
            shape = obj.shape
            dtype = str(obj.dtype)
            score = 0
            if any(k in nm for k in ("name","joint")) and obj.ndim == 1 and obj.dtype.kind in "OUS":
                score += 10
            if any(k in nm for k in ("qpos","angle","pose","mocap","joint")) and obj.ndim == 2:
                score += 8
            if any(k in nm for k in ("fps","dt","timestep","sample_rate","rate")) and obj.ndim in (0,1):
                score += 6
            if score > 0:
                cands.append((score, name, shape, dtype, obj))
    f.visititems(visit)

    # print top candidates
    cands.sort(reverse=True, key=lambda x: x[0])
    print("Top HDF5 dataset candidates:")
    for score, name, shape, dtype, obj in cands[:30]:
        print(f" [{score:02d}] /{name:<60} shape={shape} dtype={dtype}")
        sm = sample_dataset(obj)
        print(f"      sample: {short(sm, 200)}")

    # also print top-level groups to help navigation
    print("\nTop-level groups:")
    for key in f.keys():
        item = f[key]
        if isinstance(item, h5py.Group):
            print(f" /{key}  (group)  members={len(item.keys())}")
        else:
            print(f" /{key}  (dataset) shape={item.shape} dtype={item.dtype}")

# ---------- 5) Build env (factory), get env joint count for comparison ----------
header("Env+joint info (from factory)")

# Try factory with the loader if supported; otherwise bare env
fsig = inspect.signature(walk_factory)
kw = {}
for key in ("traj_generator","trajectory_loader","loader","traj","generator"):
    if key in fsig.parameters:
        kw[key] = loader
        break
if "terminal_com_dist" in fsig.parameters:
    kw["terminal_com_dist"] = float("inf")

try:
    env = walk_factory(**kw)
    print("Env created with factory ✅ kwargs:", kw)
except TypeError as e:
    print("Factory didn’t accept loader; falling back to default env. ERR:", e)
    env = walk_factory()

ph = env.physics
env_joint_names = [ph.model.id2name(i,'joint') for i in range(ph.model.njnt)]
print("Env joints:", len(env_joint_names))
print("First 20 env joint names:", env_joint_names[:20])

# ---------- 6) Suggest candidate (T×N) arrays whose N ~ #env joints ----------
header("Match candidates by column count ≈ #env joints")
nj = len(env_joint_names)

def near(a,b, tol=0.34):
    # within 34% is shown; exact matches highlighted
    return abs(a-b) <= max(1, tol*b)

print("From loader attributes:")
for nm, shape, typ in angles_cand:
    T,N = shape
    mark = "✅" if N == nj else ("~" if near(N, nj) else " ")
    print(f" {mark} {nm:30} shape={shape} (N vs env={N} vs {nj})")

print("\nFrom HDF5 datasets:")
for _, name, shape, dtype, _ in cands:
    if len(shape) == 2:
        T,N = shape
        mark = "✅" if N == nj else ("~" if near(N, nj) else " ")
        print(f" {mark} /{name:60} shape={shape} dtype={dtype}")

print("\nDone. Use the ✅ (exact) or ~ (close) candidates for joint angles; pair with a name array above.")
